# Agentic Glue ETL Pipeline - Parquet Only

This notebook demonstrates a complete Bronze → Silver → Gold ETL pipeline using:
- **Groq LLM** (via LangChain) for code generation
- **PySpark** for data processing
- **Parquet** for storage (columnar format, 5-10x compression)

## Pipeline Flow:
1. **Bronze Layer**: Raw data ingestion from CSV/JSON files
2. **Silver Layer**: Data cleaning, validation, deduplication
3. **Gold Layer**: Aggregations and business metrics
4. **Output**: Parquet files (local or S3)

## 1. Setup and Imports

In [ ]:
# Add project root to path
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

# Import our agents
from agents.code_generator_agent import CodeGeneratorAgent
from agents.validator_agent import ValidatorAgent
from agents.executor_agent import ExecutorAgent

# Other imports
import os
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

print("✅ All imports successful")

In [ ]:
# Verify Groq connection
from groq import Groq
client = Groq(api_key=os.getenv("GROQ_API_KEY"))
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": "Reply with: Groq is ready"}],
    max_tokens=20
)
print(f"✅ {response.choices[0].message.content}")

## 2. Initialize Agents

In [ ]:
# Initialize all three agents
code_gen = CodeGeneratorAgent()
validator = ValidatorAgent()
executor = ExecutorAgent()

print("✅ Agents initialized")
print(f"   - Code Generator: {code_gen.llm.model_name}")
print(f"   - Validator: Ready")
print(f"   - Executor: Ready")

## 3. Create Sample Data (For Testing)

In [ ]:
# Create sample sales data
import random
from datetime import datetime, timedelta

products = ['Laptop', 'Mouse', 'Keyboard', 'Monitor', 'Desk', 'Chair', 'Headphones']
regions = ['North', 'South', 'East', 'West']

data = []
start_date = datetime(2024, 1, 1)

for i in range(1000):
    date = start_date + timedelta(days=random.randint(0, 90))
    product = random.choice(products)
    region = random.choice(regions)
    quantity = random.randint(1, 10)
    price = round(random.uniform(10, 1000), 2)
    amount = round(quantity * price, 2)
    
    # Add some nulls and duplicates for testing
    if i % 50 == 0:
        product = None
    if i % 100 == 0:
        data.append([date, product, region, quantity, price, amount])
    
    data.append([date, product, region, quantity, price, amount])

df = pd.DataFrame(data, columns=['transaction_date', 'product', 'region', 'quantity', 'unit_price', 'total_amount'])

os.makedirs('data/raw', exist_ok=True)
df.to_csv('data/raw/sales_transactions.csv', index=False)
print(f"✅ Created sample data with {len(df):,} records")
print(f"   File: data/raw/sales_transactions.csv")
print(f"\nFirst 5 rows:")
df.head()

## 4. Bronze Layer: Raw Data Ingestion

In [ ]:
print("🤖 Generating Bronze layer code...")
bronze_code = code_gen.generate_bronze_code()

print(f"\n📝 Generated Bronze Code ({len(bronze_code)} characters):")
print("="*60)
print(bronze_code[:800] + "...\n" if len(bronze_code) > 800 else bronze_code)

In [ ]:
print("🔍 Validating Bronze code...")
bronze_validation = validator.validate(bronze_code, layer="bronze")
print(bronze_validation.summary())

print("\n🚀 Executing Bronze layer...")
bronze_result = executor.execute_code(bronze_code, "bronze")
print(bronze_result.summary())

## 5. Silver Layer: Data Quality & Cleaning

In [ ]:
print("🤖 Generating Silver layer code...")
silver_code = code_gen.generate_silver_code()

print(f"\n📝 Generated Silver Code ({len(silver_code)} characters):")
print("="*60)
print(silver_code[:800] + "...\n" if len(silver_code) > 800 else silver_code)

In [ ]:
print("🔍 Validating Silver code...")
silver_validation = validator.validate(silver_code, layer="silver")
print(silver_validation.summary())

print("\n🚀 Executing Silver layer...")
silver_result = executor.execute_code(silver_code, "silver")
print(silver_result.summary())

## 6. Gold Layer: Business Metrics & Aggregations

In [ ]:
print("🤖 Generating Gold layer code...")
gold_code = code_gen.generate_gold_code()

print(f"\n📝 Generated Gold Code ({len(gold_code)} characters):")
print("="*60)
print(gold_code[:800] + "...\n" if len(gold_code) > 800 else gold_code)

In [ ]:
print("🔍 Validating Gold code...")
gold_validation = validator.validate(gold_code, layer="gold")
print(gold_validation.summary())

print("\n🚀 Executing Gold layer...")
gold_result = executor.execute_code(gold_code, "gold")
print(gold_result.summary())

## 7. Run Complete Pipeline (Orchestrated)

In [ ]:
# Create new executor for pipeline run
pipeline_executor = ExecutorAgent()

# Run complete pipeline
pipeline_results = pipeline_executor.execute_pipeline(
    bronze_code=bronze_code,
    silver_code=silver_code,
    gold_code=gold_code,
    pipeline_name="sales_etl_pipeline"
)

# Display results
print("\n" + "="*60)
print("🏆 PIPELINE EXECUTION SUMMARY")
print("="*60)
for layer, result in pipeline_results.items():
    status = "✅ PASS" if result.success else "❌ FAIL"
    print(f"{status} - {layer.upper()}: {result.execution_time_seconds:.2f}s")
    if result.record_count:
        print(f"         Records: {result.record_count:,}")

print(pipeline_executor.get_execution_summary())

## 8. Verify Results

Check the Parquet output files created by each layer.

In [ ]:
# Check Bronze output
bronze_path = Path("data/bronze/")
if bronze_path.exists():
    print("📁 Bronze Layer Output (Parquet):")
    for parquet_dir in bronze_path.glob("*/"):
        parquet_files = list(parquet_dir.glob("*.parquet"))
        print(f"   - {parquet_dir.name}: {len(parquet_files)} parquet files")

# Check Silver output
silver_path = Path("data/silver/")
if silver_path.exists():
    print("\n📁 Silver Layer Output (Parquet):")
    for parquet_dir in silver_path.glob("*/"):
        parquet_files = list(parquet_dir.glob("*.parquet"))
        print(f"   - {parquet_dir.name}: {len(parquet_files)} parquet files")

# Check Gold output
gold_path = Path("data/gold/")
if gold_path.exists():
    print("\n📁 Gold Layer Output (Parquet):")
    for parquet_dir in gold_path.glob("*/"):
        parquet_files = list(parquet_dir.glob("*.parquet"))
        print(f"   - {parquet_dir.name}: {len(parquet_files)} parquet files")

# Check generated code audit trail
code_output_path = Path("output/generated_code/")
if code_output_path.exists():
    code_files = list(code_output_path.glob("*.py"))
    print(f"\n📝 Generated Code Files: {len(code_files)} files")
    for code_file in code_files[-3:]:
        print(f"   - {code_file.name}")

## 9. Read Parquet Outputs with PySpark

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

# Read revenue by product
revenue_by_product = spark.read.parquet("data/gold/sales_aggregated/revenue_by_product")
print("📊 Revenue by Product:")
revenue_by_product.show(10)

# Read daily trend
daily_trend = spark.read.parquet("data/gold/sales_aggregated/daily_trend")
print("\n📈 Daily Sales Trend:")
daily_trend.orderBy("date").show(10)

# Read revenue by region
revenue_by_region = spark.read.parquet("data/gold/sales_aggregated/revenue_by_region")
print("\n🌍 Revenue by Region:")
revenue_by_region.show()

## 10. Cleanup

In [ ]:
# Cleanup Spark session
executor.cleanup()
pipeline_executor.cleanup()

print("✅ Spark sessions stopped")
print("\n🎉 Agentic ETL Pipeline Complete!")
print("   - Generated code saved to: output/generated_code/")
print("   - Bronze output (Parquet): data/bronze/")
print("   - Silver output (Parquet): data/silver/")
print("   - Gold output (Parquet): data/gold/")